<a href="https://colab.research.google.com/github/fatemarampurawalaa-sudo/LLM-practice/blob/main/Tool_calling_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai

import pandas as pd
from google.colab import userdata
from google import genai
from google.genai import types

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
CHAT_MODEL = "gemini-flash-lite-latest"

print("Setup done ✅")


Setup done ✅


In [2]:
question_1 = "Calculate exactly: (1,234,567 x 0.0347) + 98,765. Give only the number."

ai_answer = client.models.generate_content(model=CHAT_MODEL, contents=question_1).text
python_answer = (1234567 * 0.0347) + 98765

print("AI answer:     ", ai_answer.strip())
print("Python answer: ", round(python_answer, 4))
print()

question_2 = "Which 2-bedroom flats does our company currently have available in JVC, and at what prices?"
print("AI answer:", client.models.generate_content(model=CHAT_MODEL, contents=question_2).text)

AI answer:      141,644.0769
Python answer:  141604.4749

AI answer: I do not have access to your company's internal inventory, database, or CRM system, so I cannot tell you which specific 2-bedroom flats are currently available in JVC or their prices. 

If you can provide a list of the available properties, paste a spreadsheet, or upload the inventory data here, I would be happy to organize, summarize, or format that information for you!


In [3]:
def calculate_yield(price: float, annual_rent: float, service_charge: float = 0) -> dict:
    """Calculate the rental yield of a property.

    Args:
        price: Purchase price in AED.
        annual_rent: Yearly rent in AED.
        service_charge: Yearly service charge in AED (0 if unknown).
    """
    print(f"🔧 Tool used: calculate_yield(price={price}, annual_rent={annual_rent}, service_charge={service_charge})")
    gross = annual_rent / price * 100
    net = (annual_rent - service_charge) / price * 100
    return {"gross_yield_percent": round(gross, 2), "net_yield_percent": round(net, 2)}


response = client.models.generate_content(
    model=CHAT_MODEL,
    contents="I'm buying a flat for 1.2 million dirhams. It rents for 85k a year and the service charge is 18k. What's my yield?",
    config=types.GenerateContentConfig(tools=[calculate_yield]),
)

print()
print("AI answer:", response.text)

🔧 Tool used: calculate_yield(price=1200000, annual_rent=85000, service_charge=18000)

AI answer: Your gross rental yield is **7.08%** and your net rental yield (after accounting for the 18,000 AED service charge) is **5.58%**.


In [4]:
properties = pd.DataFrame({
    "id":       [101, 102, 103, 104, 105, 106, 107, 108],
    "area":     ["JVC", "JVC", "Marina", "Marina", "Downtown", "JVC", "Business Bay", "Marina"],
    "beds":     [1, 2, 2, 3, 2, 2, 1, 1],
    "price":    [650000, 880000, 1450000, 2300000, 2600000, 920000, 1100000, 990000],
    "annual_rent": [55000, 68000, 98000, 150000, 160000, 70000, 85000, 75000],
})

def search_properties(area: str, max_price: float = 0, beds: int = 0) -> list[dict]:
    """Search the company's available properties.

    Args:
        area: Area name, e.g. JVC, Marina, Downtown, Business Bay.
        max_price: Maximum price in AED. Use 0 for no limit.
        beds: Number of bedrooms. Use 0 for any.
    """
    print(f"🔧 Tool used: search_properties(area={area}, max_price={max_price}, beds={beds})")
    result = properties[properties["area"].str.lower() == area.lower()]
    if max_price > 0:
        result = result[result["price"] <= max_price]
    if beds > 0:
        result = result[result["beds"] == beds]
    return result.to_dict(orient="records")


response = client.models.generate_content(
    model=CHAT_MODEL,
    contents="Do you have any 2 bedroom flats in JVC under 900k?",
    config=types.GenerateContentConfig(tools=[search_properties]),
)

print()
print("AI answer:", response.text)

🔧 Tool used: search_properties(area=JVC, max_price=900000, beds=2)

AI answer: Yes, we do have a 2-bedroom flat available in JVC under 900k:

*   **Price:** AED 880,000
*   **Bedrooms:** 2
*   **Area:** JVC
*   **Current Annual Rent (if applicable):** AED 68,000

Would you like to schedule a viewing or get more details about this property?


In [5]:
questions = [
    "What's the yield on a 950k flat renting for 72k a year?",
    "What do you have in Marina with 2 bedrooms?",
    "Hi! What can you help me with?",
]

for q in questions:
    print("Q:", q)
    response = client.models.generate_content(
        model=CHAT_MODEL,
        contents=q,
        config=types.GenerateContentConfig(tools=[calculate_yield, search_properties]),
    )
    print("A:", response.text)
    print("-" * 60)

Q: What's the yield on a 950k flat renting for 72k a year?
🔧 Tool used: calculate_yield(price=950000, annual_rent=72000, service_charge=0)
A: The gross rental yield for that property is **7.58%**.
------------------------------------------------------------
Q: What do you have in Marina with 2 bedrooms?
🔧 Tool used: search_properties(area=Marina, max_price=0, beds=2)
A: We currently have 1 two-bedroom property available in Dubai Marina:

* **Price:** AED 1,450,000
* **Expected Annual Rent:** AED 98,000

Would you like more details about this property or help calculating the potential rental yield?
------------------------------------------------------------
Q: Hi! What can you help me with?
A: Hello! I'm your real estate assistant for Dubai properties. I can help you with several things:

1. **Search for properties:** I can search our listings by area (like JVC, Marina, Downtown, or Business Bay), number of bedrooms, and maximum price.
2. **Calculate rental yields:** If you have a purc

In [6]:
task = "Find 2-bedroom flats in JVC under 1 million and tell me which one has the best gross yield."

response = client.models.generate_content(
    model=CHAT_MODEL,
    contents=task,
    config=types.GenerateContentConfig(
        tools=[calculate_yield, search_properties],
        system_instruction="You are a property assistant. Always use the calculate_yield tool for any yield calculation. Never calculate yields yourself.",
        automatic_function_calling=types.AutomaticFunctionCallingConfig(maximum_remote_calls=6),
    ),
)

print()
print("AI answer:", response.text)

🔧 Tool used: search_properties(area=JVC, max_price=1000000, beds=2)
🔧 Tool used: calculate_yield(price=880000, annual_rent=68000, service_charge=0)
🔧 Tool used: calculate_yield(price=920000, annual_rent=70000, service_charge=0)

AI answer: I found two 2-bedroom flats in JVC under 1 million AED:

1. **Property ID 102**
   - **Price:** AED 880,000
   - **Annual Rent:** AED 68,000
   - **Gross Yield:** 7.73%

2. **Property ID 106**
   - **Price:** AED 920,000
   - **Annual Rent:** AED 70,000
   - **Gross Yield:** 7.61%

**Property ID 102** offers the best gross yield at **7.73%**.


In [7]:
def calculate_yield_safe(price: float, annual_rent: float, service_charge: float = 0) -> dict:
    """Calculate the rental yield of a property.

    Args:
        price: Purchase price in AED.
        annual_rent: Yearly rent in AED.
        service_charge: Yearly service charge in AED (0 if unknown).
    """
    print(f"🔧 Tool used: calculate_yield_safe(price={price}, annual_rent={annual_rent})")
    if price <= 0 or annual_rent <= 0:
        return {"error": "Price and rent must be positive numbers."}
    gross = annual_rent / price * 100
    net = (annual_rent - service_charge) / price * 100
    return {"gross_yield_percent": round(gross, 2), "net_yield_percent": round(net, 2)}


def request_property_removal(property_id: int) -> dict:
    """Request removal of a property listing. Removal needs approval from a manager.

    Args:
        property_id: The ID of the property to remove.
    """
    print(f"🔧 Tool used: request_property_removal(property_id={property_id})")
    return {
        "status": "pending_approval",
        "message": f"Property {property_id} was NOT removed. A manager must approve this request.",
    }


tests = [
    "What's the yield on a flat priced at -500000 with rent 40000?",
    "Please remove property 102 from the listings.",
]

for q in tests:
    print("Q:", q)
    response = client.models.generate_content(
        model=CHAT_MODEL,
        contents=q,
        config=types.GenerateContentConfig(
            tools=[calculate_yield_safe, request_property_removal],
            automatic_function_calling=types.AutomaticFunctionCallingConfig(maximum_remote_calls=3),
        ),
    )
    print("A:", response.text)
    print("-" * 60)

Q: What's the yield on a flat priced at -500000 with rent 40000?
🔧 Tool used: calculate_yield_safe(price=-500000, annual_rent=40000)
A: The price cannot be negative. Please provide a valid, positive purchase price to calculate the rental yield.
------------------------------------------------------------
Q: Please remove property 102 from the listings.
🔧 Tool used: request_property_removal(property_id=102)
A: I have submitted a request to remove property 102 from the listings. However, since manager approval is required, the property has not been removed yet.
------------------------------------------------------------
